# 02 — Building the Analytical Datasets

This notebook turns the 12 cleaned CSVs into the **3 analytical tables** everything else
(EDA, ML, Power BI) will run on:

- `order_analytics.csv` — one row per order
- `customer_analytics.csv` — one row per customer, with a churn label
- `restaurant_analytics.csv` — one row per restaurant

All the actual logic lives in `src/ingest.py`, `src/clean.py`, and `src/features.py` as
plain, commented functions. This notebook just calls them in order and shows what came out,
so you can see and check each step rather than trusting a black box.

## Step 1 — Load the raw (cleaned) CSVs

In [1]:
import sys
sys.path.insert(0, '../src')

from ingest import load_cleaned
from features import (
    orders_ready_for_analytics,
    build_order_analytics, build_customer_analytics, build_restaurant_analytics,
)

CLEANED_DIR = "../data/cleaned/"   # change this if your cleaned CSVs live somewhere else

data = load_cleaned(CLEANED_DIR)
for name, df in data.items():
    print(f"{name:20s} {df.shape}")

cities               (25, 5)
customers            (12000, 15)
restaurants          (1200, 13)
delivery_partners    (2000, 10)
menu                 (8997, 8)
promotions           (300, 6)
orders               (20500, 15)
order_items          (42378, 6)
payments             (20291, 6)
customer_feedback    (13970, 7)
weather              (18173, 7)
traffic              (18335, 6)


## Step 2 — Light cleaning before joining

`orders_ready_for_analytics` drops the handful of orders (~25 out of 20,500) that
have no valid CustomerID or RestaurantID — those can't be linked to anything, so
they'd just create blank rows in every join below. (This is separate from
`src/clean.py`, which cleans the raw data further upstream.)

In [2]:
orders = orders_ready_for_analytics(data["orders"])
print("orders before:", len(data["orders"]))
print("orders after: ", len(orders))

orders before: 20500
orders after:  20475


## Step 3 — Build `order_analytics.csv`

One row per order. Brings together customer info, restaurant info, delivery-partner
info, weather, traffic, and order-item totals for every order.

**Known limitation:** customers have no latitude/longitude in the source data (only
restaurants do), so we can't calculate a real delivery distance. We use `SameCity`
(customer city == restaurant city) as a documented stand-in instead of making up
coordinates — see `documentation/feature_dictionary.md` for the full writeup.

In [3]:
order_analytics = build_order_analytics(data, orders)
print(order_analytics.shape)
order_analytics.head()

(20475, 74)


,OrderID,OrderDate,OrderTime,OrderHour,DayOfWeek,Month,Quarter,Year,IsWeekend,PeakHour,...,UniqueItems,ItemRevenue,AverageItemPrice,RestaurantPreparationTime,BasketSize,CustomerRating,DeliveryRating,FoodRating,Sentiment,SameCity
0,115576,2023-04-25,11:33:00,11,Tuesday,4.0,2.0,2023.0,0,0,...,1,606.0,202.000000,11.000000,3,NaN,NaN,NaN,NaN,0
1,103143,2023-07-17,10:42:00,10,Monday,7.0,3.0,2023.0,0,0,...,1,325.0,325.000000,17.000000,1,5.0,5.0,5.0,Positive,0
2,102038,2024-04-01,17:07:00,17,Monday,4.0,2.0,2024.0,0,0,...,2,1108.0,277.000000,24.500000,4,5.0,4.0,5.0,Positive,0
3,113029,2024-10-05,12:30:00,12,Saturday,10.0,4.0,2024.0,1,1,...,2,1848.0,301.000000,22.333333,6,5.0,5.0,5.0,Positive,0
4,116948,2024-10-13,15:47:00,15,Sunday,10.0,4.0,2024.0,1,0,...,2,704.0,117.333333,21.333333,6,3.0,3.0,4.0,Neutral,0


## Step 4 — Build `customer_analytics.csv` (time-aware churn)

This one needs extra care. If we just label "no order in the last 60 days" as churned
and then split the data randomly for ML, the model would be trained partly on
information from the future — that's a classic data leakage bug.

Instead:
1. Pick an **observation date** = the last order date in the whole dataset, minus 60 days.
2. Every feature (order count, spend, ratings, recency...) is computed using
   **only orders on or before that date.**
3. `Churn60` = 1 if the customer placed **no** order in the 60 days *after* that date.

This way the label always looks forward from a fixed point in time, the same way it
would need to work in a real production system.

In [4]:
customer_analytics = build_customer_analytics(data, orders)
print(customer_analytics.shape)
print("Churn rate:", customer_analytics['Churn60'].mean().round(3))
customer_analytics.head()

(9254, 24)
Churn rate: 0.878


,CustomerID,Age,Gender,City,Membership,RegistrationDate,TotalOrders,CustomerLifetimeValue,AverageOrderValue,AverageDeliveryTime,...,OrderFrequency,OrdersLast30Days,OrdersLast60Days,OrdersPrevious60Days,CouponUsageRate,CancellationRate,WeekendOrderRate,PreferredCuisine,CustomerTenure,Churn60
0,1,35.0,Male,Nagpur,Basic,2022-06-11,1,744.50,744.500,14.0,...,0.001144,0,0,0,0.0,1.0,0.0,Fast Food,874.0,1
1,2,29.0,Male,Hyderabad,Zomato Pro,2024-02-18,2,810.95,405.475,31.5,...,0.007782,0,0,0,0.5,0.0,0.0,Mughlai,257.0,1
2,3,26.0,Other,Vadodara,Gold,2023-05-17,2,966.50,483.250,56.5,...,0.003745,2,2,0,0.5,0.0,1.0,Mughlai,534.0,1
3,6,NaN,Other,Bengaluru,Basic,2024-02-28,2,840.85,420.425,39.0,...,0.008097,0,0,0,0.0,0.5,1.0,Street Food,247.0,1
4,7,28.0,Female,Hyderabad,Basic,2023-08-25,1,664.75,664.750,37.0,...,0.002304,0,0,0,0.0,1.0,0.0,Chinese,434.0,1


**Note on the churn rate:** it comes out around 88%, i.e. heavily imbalanced.
That's expected here — most customers only order ~1-2 times across the full 2-year window,
so a 60-day window naturally catches most of them as "no order". Keep this in mind for the
churn model step — you'll want class weighting or resampling (e.g. `class_weight='balanced'`,
or SMOTE), not accuracy as your main metric (use F1 / ROC-AUC instead).

## Step 5 — Build `restaurant_analytics.csv`

In [5]:
restaurant_analytics = build_restaurant_analytics(data, orders)
print(restaurant_analytics.shape)
restaurant_analytics.head()

(1200, 20)


,RestaurantID,RestaurantName,City,Cuisine,RestaurantType,Rating,AverageCost,TotalOrders,CompletedOrders,CancelledOrders,CancellationRate,TotalRevenue,AverageOrderValue,AverageDeliveryTime,AverageCustomerRating,LateDeliveryRate,RestaurantPopularity,RevenueRankCity,PerformanceScore,PerformanceCategory
0,1,Curry Cafe,Vadodara,Healthy Food,Delivery Only,3.4,328.0,17,12,4,0.2353,6325.90,372.111765,44.312500,3.545455,0.2353,51.2,23,0.4543,Poor
1,2,Curry Treats,Surat,Burgers,Delivery Only,NaN,525.0,18,11,4,0.2222,7060.45,392.247222,37.000000,3.333333,0.0556,60.2,18,0.5533,Good
2,3,Blue Kitchen,Nashik,Healthy Food,Cloud Kitchen,4.3,171.0,16,12,3,0.1875,6229.75,389.359375,40.133333,3.333333,0.1250,41.9,24,0.5869,Good
3,4,The Grill,Nashik,Biryani,Dine-in & Delivery,3.3,258.0,20,10,5,0.2500,8402.85,420.142500,38.210526,3.857143,0.1500,77.2,7,0.5108,Average
4,5,Golden Diner,Pune,Chinese,Dine-in,3.3,529.0,15,9,6,0.4000,5791.95,386.130000,38.933333,3.444444,0.0667,32.4,37,0.3895,Poor


## Step 6 — Quick sanity checks

Nothing fancy — just the basic things that should always be true before we trust
these tables for EDA or ML.

In [6]:
assert order_analytics['OrderID'].is_unique, "Duplicate OrderIDs!"
assert customer_analytics['CustomerID'].is_unique, "Duplicate CustomerIDs!"
assert restaurant_analytics['RestaurantID'].is_unique, "Duplicate RestaurantIDs!"
assert set(customer_analytics['Churn60'].unique()) <= {0, 1}, "Churn60 should only be 0/1"
assert (order_analytics['FinalAmount'].dropna() >= 0).all(), "Negative FinalAmount found!"

print("All checks passed.")

All checks passed.


## Step 7 — Save the analytical datasets

In [7]:
import os
os.makedirs("../data/analytical", exist_ok=True)

order_analytics.to_csv("../data/analytical/order_analytics.csv", index=False)
customer_analytics.to_csv("../data/analytical/customer_analytics.csv", index=False)
restaurant_analytics.to_csv("../data/analytical/restaurant_analytics.csv", index=False)

print("Saved all 3 analytical CSVs to ../data/analytical/")

Saved all 3 analytical CSVs to ../data/analytical/


## Where to go from here

With these three tables saved, the next notebooks branch off in parallel:

- `03_eda.ipynb` — 30 charts, all built from `order_analytics.csv`
- `04_delivery_time_model.ipynb` — regression, target = `DeliveryTimeTarget` in `order_analytics.csv`
- `05_churn_model.ipynb` — classification, target = `Churn60` in `customer_analytics.csv`

If anything looks off in these tables, check `documentation/feature_dictionary.md` first —
every column's definition and source is listed there.